In [0]:
--Ecrire des notebooks de test :
--Créez un notebook afin de tester la
--bonne historisation de quelques tables
--de votre pipeline, notamment
--sales_order_detail.

USE CATALOG barbara_lakehouse;

SHOW SCHEMAS;

DESCRIBE TABLE barbara_lakehouse.silver.sales_order_detail;

-- Vérifier les lignes actives
SELECT COUNT(*) AS nb_actifs
FROM barbara_lakehouse.silver.sales_order_detail
WHERE _tf_valid_to IS NULL;

-- Une seule ligne active par clé métier
SELECT
  sales_order_id,
  sales_order_detail_id,
  COUNT(*) AS nb_versions_actives
FROM barbara_lakehouse.silver.sales_order_detail
WHERE _tf_valid_to IS NULL
GROUP BY sales_order_id, sales_order_detail_id
HAVING COUNT(*) > 1;

-- Lignes actives Bronze/Silver
SELECT
  (SELECT COUNT(*) FROM barbara_lakehouse.bronze.salesorderdetail) AS bronze_cnt,
  (SELECT COUNT(*) FROM barbara_lakehouse.silver.sales_order_detail WHERE _tf_valid_to IS NULL) AS silver_active_cnt;

-- Historique existant
SELECT COUNT(*) AS nb_versions_cloturees
FROM barbara_lakehouse.silver.sales_order_detail
WHERE _tf_valid_to IS NOT NULL;

-- Trouver plusieurs versions pour une clé
SELECT
  sales_order_id,
  sales_order_detail_id,
  COUNT(*) AS nb_versions
FROM barbara_lakehouse.silver.sales_order_detail
GROUP BY sales_order_id, sales_order_detail_id
HAVING COUNT(*) >= 2
ORDER BY nb_versions DESC
LIMIT 20;